# 04 — Gemma 3 VLM Zero-Shot Cloud Estimation

## Executive Summary
**Method:** Gemma 3 (`gemma3:4b`) running locally via Ollama.  
**Input:** One image per day (~12:00 UTC) with dome mask applied (~180 images from pilot).  
**Output:** `cloud_fraction` [0–1], `cloud_type`, `confidence` — parsed from structured JSON.

## Exact prompt used (scientifically important — cite this in any publication)

```
You are an expert meteorologist analysing a fish-eye sky camera image
taken at a weather station in Warsaw, Poland.

Examine the image carefully and estimate:
1. The fraction of the sky dome covered by clouds (0.0 = completely clear, 1.0 = completely overcast).
2. The dominant cloud type visible.
3. Your confidence in the estimate.

Respond ONLY with a valid JSON object in exactly this format, with no other text:
{
  "cloud_fraction": <float between 0.0 and 1.0>,
  "cloud_type": "<one of: clear / cumulus / stratus / cirrus / cumulonimbus / mixed>",
  "confidence": "<one of: low / medium / high>"
}
```

**Note:** Results depend on the exact model version and prompt — both are logged
in `outputs/csv/cf_gemma_raw.csv` for full reproducibility.

In [1]:
import sys, logging
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(message)s',
                    datefmt='%H:%M:%S')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skycamera.config import CX, CY, R, CSV_DIR, PLOTS_DIR, RAW_DIR
from skycamera.io import load_image, build_image_index
from skycamera.preprocessing import build_circular_mask
from skycamera.vlm import (
    check_ollama, query_vlm, run_vlm_on_index,
    PROMPT_TEMPLATE, DEFAULT_MODEL
)

RAW_ROOT  = RAW_DIR   # data/raw/ inside project
CF_VLM    = CSV_DIR / 'cf_gemma.csv'
CF_VLM_RAW = CSV_DIR / 'cf_gemma_raw.csv'   # full audit trail incl. raw text

CSV_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Dome mask
_s = load_image(RAW_ROOT / '2024-01-15' / '2024_01_15__12_04_51.jpg')
dome_mask = build_circular_mask(_s.shape[0], _s.shape[1], CX, CY, R)
print(f'Setup complete  |  dome pixels: {dome_mask.sum():,}')

Setup complete  |  dome pixels: 2,762,359


## 1. Check Ollama server and model availability

In [2]:
status = check_ollama(model=DEFAULT_MODEL)
print(f'Server reachable  : {status["server_ok"]}')
print(f'Model available   : {status["model_available"]}')
print(f'Available models  : {status["available_models"]}')

if not status['server_ok']:
    raise RuntimeError(
        'Ollama server is not running.\n'
        'Start it with: ollama serve\n'
        'Or just open the Ollama desktop app.'
    )
if not status['model_available']:
    raise RuntimeError(
        f'Model {DEFAULT_MODEL!r} not found.\n'
        f'Pull it with: ollama pull {DEFAULT_MODEL}\n'
        f'Available: {status["available_models"]}'
    )

Server reachable  : True
Model available   : True
Available models  : ['gemma3:4b']


## 2. Single-image test — validate prompt and JSON parsing

Before running the full batch, verify the model responds in the expected format.

In [ ]:
from skycamera.preprocessing import apply_mask

test_paths = {
    'January (overcast)': RAW_ROOT / '2024-01-15' / '2024_01_15__12_04_51.jpg',
    'June (clear)':       RAW_ROOT / '2024-06-15' / '2024_06_15__12_00_31.jpg',
}

for label, fpath in test_paths.items():
    img = load_image(fpath)
    img_masked = apply_mask(img, dome_mask)
    result = query_vlm(img_masked, model=DEFAULT_MODEL)
    print(f'\n{label}:')
    print(f'  cloud_fraction : {result["cloud_fraction"]}')
    print(f'  cloud_type     : {result["cloud_type"]}')
    print(f'  confidence     : {result["confidence"]}')
    print(f'  raw_response   : {result["raw_response"][:120]}')

16:37:24 | VLM attempt 1 failed (timed out), retrying...



January (overcast):
  cloud_fraction : 0.65
  cloud_type     : cumulus
  confidence     : medium
  raw_response   : ```json
{
  "cloud_fraction": 0.65,
  "cloud_type": "cumulus",
  "confidence": "medium"
}
```


## 3. Visual check — show test images alongside VLM estimate

In [ ]:
fig, axes = plt.subplots(1, len(test_paths), figsize=(10, 5))
for ax, (label, fpath) in zip(axes, test_paths.items()):
    img = load_image(fpath)
    masked = apply_mask(img, dome_mask)
    result = query_vlm(masked, model=DEFAULT_MODEL)
    ax.imshow(masked)
    ax.set_title(
        f'{label}\nCF={result["cloud_fraction"]}  {result["cloud_type"]}\n'
        f'confidence={result["confidence"]}',
        fontsize=9
    )
    ax.axis('off')
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'vlm_test_images.png', bbox_inches='tight', dpi=100)
plt.show()

## 4. Full batch inference — one image per day

~180 images from the 2024 pilot dataset (one per day, closest to 12:00 UTC).

**Expected runtime:** ~30–90 seconds per image on CPU → 1.5–4.5 hours total.
Progress is saved after every image to `cf_gemma_raw.csv` so the run can be
interrupted and resumed safely.

In [ ]:
INDEX_CSV = CSV_DIR / 'image_index.csv'

if INDEX_CSV.exists():
    df_index = pd.read_csv(INDEX_CSV, parse_dates=['timestamp'])
    df_index['path'] = df_index['path'].apply(Path)
else:
    df_index = build_image_index(RAW_ROOT, apply_daytime_filter=True)
    df_index.to_csv(INDEX_CSV, index=False)

print(f'Index loaded: {df_index["is_daytime"].sum():,} daytime images')

if CF_VLM.exists():
    print(f'Loading cached VLM results from {CF_VLM}...')
    df_vlm = pd.read_csv(CF_VLM, parse_dates=['timestamp'])
    print(f'Loaded {len(df_vlm)} records')
else:
    print(f'Starting VLM inference with {DEFAULT_MODEL}...')
    print('(Progress is saved after every image — safe to interrupt with Ctrl+C)')
    # Note: VLM can process night images. run_vlm_on_index selects
# one image per day closest to 12:00 UTC (effectively daytime).
df_vlm = run_vlm_on_index(
        df_index, dome_mask,
        model=DEFAULT_MODEL,
        subsample='1D',
        timeout=120,
        save_raw=CF_VLM_RAW,
    )
    df_vlm.to_csv(CF_VLM, index=False)
    print(f'Saved -> {CF_VLM}')

print(f'\nVLM results: {len(df_vlm)} images')
print(f'Parse failures (CF=None): {df_vlm["cloud_fraction"].isna().sum()}')
df_vlm.head()

## 5. Results analysis

In [ ]:
df_ok = df_vlm.dropna(subset=['cloud_fraction'])
print(f'Valid CF estimates: {len(df_ok)} / {len(df_vlm)}')
print(f'Mean CF: {df_ok["cloud_fraction"].mean():.3f}')
print()
print('Cloud type distribution:')
print(df_ok['cloud_type'].value_counts().to_string())
print()
print('Confidence distribution:')
print(df_ok['confidence'].value_counts().to_string())

In [ ]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time series
axes[0,0].scatter(df_ok['timestamp'], df_ok['cloud_fraction'],
                  s=20, alpha=0.6, color='steelblue')
axes[0,0].set_ylabel('Cloud fraction'); axes[0,0].set_title('VLM cloud fraction time series')
axes[0,0].set_ylim(-0.05, 1.05); axes[0,0].grid(alpha=0.2)

# Monthly boxplot
monthly_groups = [df_ok[df_ok['month'] == m]['cloud_fraction'].values
                  for m in range(1, 13)]
bp = axes[0,1].boxplot(monthly_groups, labels=month_names,
                       patch_artist=True, showfliers=False)
for patch in bp['boxes']:
    patch.set_facecolor('steelblue'); patch.set_alpha(0.6)
axes[0,1].set_ylabel('Cloud fraction'); axes[0,1].set_title('Monthly distribution (VLM)')
axes[0,1].set_ylim(-0.05, 1.05); axes[0,1].grid(axis='y', alpha=0.3)

# Cloud type bar chart
ct = df_ok['cloud_type'].value_counts()
axes[1,0].bar(ct.index, ct.values, color='steelblue', edgecolor='white')
axes[1,0].set_xlabel('Cloud type'); axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Cloud type frequency')
axes[1,0].tick_params(axis='x', rotation=20)

# Confidence pie
conf = df_ok['confidence'].value_counts()
axes[1,1].pie(conf.values, labels=conf.index, autopct='%1.0f%%',
              colors=['#2c7bb6','#fdae61','#d7191c'])
axes[1,1].set_title('Confidence distribution')

fig.tight_layout()
fig.savefig(PLOTS_DIR / 'vlm_results.png', bbox_inches='tight', dpi=100)
plt.show()

## 6. Compare VLM vs R/B threshold

In [ ]:
CF_RB_CSV = CSV_DIR / 'cf_rb_threshold.csv'

if not CF_RB_CSV.exists():
    print('R/B results not found — run notebook 02 first.')
else:
    df_rb = pd.read_csv(CF_RB_CSV, parse_dates=['timestamp'])

    # Match by date (VLM is daily, R/B is 5-min)
    df_ok_copy = df_ok.copy()
    df_ok_copy['date'] = pd.to_datetime(df_ok_copy['timestamp']).dt.date

    # Daily mean of R/B
    df_rb['date'] = pd.to_datetime(df_rb['timestamp']).dt.date
    rb_daily = df_rb.groupby('date')['cloud_fraction'].mean().reset_index()
    rb_daily.columns = ['date', 'cf_rb']

    merged = pd.merge(df_ok_copy[['date','cloud_fraction']].rename(columns={'cloud_fraction':'cf_vlm'}),
                      rb_daily, on='date', how='inner')

    ok = merged['cf_vlm'].notna() & merged['cf_rb'].notna()
    mae = float(np.mean(np.abs(merged.loc[ok,'cf_vlm'] - merged.loc[ok,'cf_rb'])))
    r   = float(np.corrcoef(merged.loc[ok,'cf_vlm'], merged.loc[ok,'cf_rb'])[0,1])
    print(f'Paired days: {ok.sum()}')
    print(f'VLM vs R/B: MAE={mae:.4f}  Pearson r={r:.4f}')

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(merged.loc[ok,'cf_rb'], merged.loc[ok,'cf_vlm'],
               alpha=0.6, color='steelblue', s=40)
    ax.plot([0,1],[0,1],'k--', linewidth=1, label='1:1')
    ax.set_xlabel('R/B threshold (daily mean)')
    ax.set_ylabel('Gemma VLM estimate')
    ax.set_title(f'R/B vs VLM  (r={r:.3f}  MAE={mae:.3f})')
    ax.legend(); ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.05,1.05)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / 'vlm_vs_rb.png', bbox_inches='tight', dpi=100)
    plt.show()

## 7. Prompt and model metadata (cite in paper)

In [ ]:
print('=== VLM Methodology Record ===')
print(f'Model: {DEFAULT_MODEL}')
print(f'Backend: Ollama (local inference, no cloud API)')
print(f'Subsampling: 1 image per day, closest to 12:00 UTC')
print(f'Images processed: {len(df_ok)}')
print(f'Parse failure rate: {df_vlm["cloud_fraction"].isna().mean()*100:.1f}%')
print()
print('Exact prompt used:')
print('-' * 50)
print(PROMPT_TEMPLATE)
print('-' * 50)
print()
print(f'Full raw responses saved to: {CF_VLM_RAW}')